In [7]:
import pandas as pd
df = pd.read_csv('dataset.csv')

In [8]:
df[(df['date']) > '2017-01-01']

,date,store,item,sales
1462,2017-01-02,1,1,15
1463,2017-01-03,1,1,10
1464,2017-01-04,1,1,16
1465,2017-01-05,1,1,14
1466,2017-01-06,1,1,24
...,...,...,...,...
912995,2017-12-27,10,50,63
912996,2017-12-28,10,50,59
912997,2017-12-29,10,50,74
912998,2017-12-30,10,50,62


In [2]:
print(df['store'].unique())
print(df['item'].unique())
df.columns


[ 1  2  3  4  5  6  7  8  9 10]
[ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24
 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48
 49 50]


Index(['date', 'store', 'item', 'sales'], dtype='object')

In [3]:
df.head()

,date,store,item,sales
0,2013-01-01,1,1,13
1,2013-01-02,1,1,11
2,2013-01-03,1,1,14
3,2013-01-04,1,1,13
4,2013-01-05,1,1,10


In [4]:
resultado = (
    df.groupby(['store', 'item'])['sales']
    .sum()
    .reset_index()
    .sort_values(['store', 'sales'], ascending=[True, False])
)

top_por_loja = resultado.groupby('store').head(1)

print(top_por_loja)

     store  item   sales
14       1    15  145497
77       2    28  205677
114      3    15  183373
164      4    15  169186
214      5    15  122319
264      6    15  121737
314      7    15  111640
364      8    15  197295
414      9    15  170069
464     10    15  180757


In [5]:
import pandas as pd
import xgboost as xgb
from sklearn.metrics import mean_absolute_error

def add_features(df):
    df = df.copy()
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values(['store', 'item', 'date'])

    df['dow'] = df['date'].dt.dayofweek
    df['month'] = df['date'].dt.month

    for l in [7, 14, 28]:
        df[f'lag_{l}'] = df.groupby(['store', 'item'])['sales'].shift(l)

    df['rmean_7'] = df.groupby(['store', 'item'])['sales'].shift(1).rolling(7).mean()
    df['rmean_28'] = df.groupby(['store', 'item'])['sales'].shift(1).rolling(28).mean()

    return df

df = add_features(df)

train = df[df['date'] < '2017-01-01'].dropna()
val = df[df['date'] >= '2017-01-01'].dropna()


features = [c for c in df.columns if c not in ['date', 'sales']]

X_train = train[features]
y_train = train['sales']

X_val = val[features]
y_val = val['sales']

model = xgb.XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=100
)

y_pred = model.predict(X_val)

print(f'Erro médio absoluto: {mean_absolute_error(y_val, y_pred)}')

[0]	validation_0-rmse:31.05003
[100]	validation_0-rmse:8.35088
[200]	validation_0-rmse:8.23594
[300]	validation_0-rmse:8.20091
[400]	validation_0-rmse:8.18440
[499]	validation_0-rmse:8.17439
Erro médio absoluto: 6.273967266082764


In [ ]:
import pandas as pd
import xgboost as xgb
from sklearn.metrics import mean_absolute_error

def add_features(df):
    df = df.copy()
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values(['store', 'item', 'date'])
    df['dow'] = df['date'].dt.dayofweek
    df['month'] = df['date'].dt.month

    for l in [7, 14, 28]:
        df[f'lag_{l}'] = df.groupby(['store', 'item'])['sales'].shift(l)

    df['rmean_7'] = df.groupby(['store', 'item'])['sales'].shift(1).rolling(7).mean()
    df['rmean_28'] = df.groupby(['store', 'item'])['sales'].shift(1).rolling(28).mean()
    return df

df = add_features(df)

train_full = df[df['date'] < '2017-01-01'].dropna()
val_full = df[df['date'] >= '2017-01-01'].dropna()


features = [c for c in df.columns if c not in ['date', 'sales', 'store']]

results = {}
total_mae = 0

for store_id in df['store'].unique():
    print(f"Treinando modelo para Loja: {store_id}")

    train_store = train_full[train_full['store'] == store_id]
    val_store = val_full[val_full['store'] == store_id]

    X_train, y_train = train_store[features], train_store['sales']
    X_val, y_val = val_store[features], val_store['sales']

    model = xgb.XGBRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )

    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False
    )

    y_pred = model.predict(X_val)
    mae = mean_absolute_error(y_val, y_pred)

    results[store_id] = mae
    total_mae += mae
    print(f"MAE Loja {store_id}: {mae:.4f}")

print("-" * 30)
print(f"MAE Médio entre todas as lojas: {total_mae / len(results):.4f}")

Treinando modelo para Loja: 1
MAE Loja 1: 6.0670
Treinando modelo para Loja: 2
MAE Loja 2: 7.2487
Treinando modelo para Loja: 3
MAE Loja 3: 6.7962
Treinando modelo para Loja: 4
MAE Loja 4: 6.6157
Treinando modelo para Loja: 5
MAE Loja 5: 5.5377
Treinando modelo para Loja: 6
MAE Loja 6: 5.4885
Treinando modelo para Loja: 7
MAE Loja 7: 5.2271
Treinando modelo para Loja: 8
MAE Loja 8: 7.1643
Treinando modelo para Loja: 9
MAE Loja 9: 6.5912
Treinando modelo para Loja: 10
MAE Loja 10: 6.7708
------------------------------
MAE Médio entre todas as lojas: 6.3507


In [6]:
import pandas as pd
import xgboost as xgb
from sklearn.metrics import mean_absolute_error

def add_features(df):
    df = df.copy()
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values(['store', 'item', 'date'])
    df['dow'] = df['date'].dt.dayofweek
    df['month'] = df['date'].dt.month

    for l in [7, 14, 28]:
        df[f'lag_{l}'] = df.groupby(['store', 'item'])['sales'].shift(l)

    df['rmean_7'] = df.groupby(['store', 'item'])['sales'].shift(1).rolling(7).mean()
    df['rmean_28'] = df.groupby(['store', 'item'])['sales'].shift(1).rolling(28).mean()
    return df

df = add_features(df)

train_full = df[df['date'] < '2017-01-01'].dropna()
val_full = df[df['date'] >= '2017-01-01'].dropna()

features = [c for c in df.columns if c not in ['date', 'sales', 'store']]

all_predictions = []
results = {}

for store_id in df['store'].unique():
    print(f"Treinando modelo para Loja: {store_id}")

    train_store = train_full[train_full['store'] == store_id]
    val_store = val_full[val_full['store'] == store_id].copy()

    X_train, y_train = train_store[features], train_store['sales']
    X_val, y_val = val_store[features], val_store['sales']

    model = xgb.XGBRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )

    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

    val_store['prediction'] = model.predict(X_val)

    # Armazena apenas as colunas desejadas para o CSV
    all_predictions.append(val_store[['date', 'item', 'store', 'prediction']])

    mae = mean_absolute_error(y_val, val_store['prediction'])
    results[store_id] = mae

# Consolidar e exportar
df_final = pd.concat(all_predictions)
df_final.to_csv('previsoes_vendas.csv', index=False)

print("-" * 30)
print(f"Arquivo 'previsoes_vendas.csv' gerado com {len(df_final)} linhas.")
print(f"MAE Médio: {sum(results.values()) / len(results):.4f}")

Treinando modelo para Loja: 1
Treinando modelo para Loja: 2
Treinando modelo para Loja: 3
Treinando modelo para Loja: 4
Treinando modelo para Loja: 5
Treinando modelo para Loja: 6
Treinando modelo para Loja: 7
Treinando modelo para Loja: 8
Treinando modelo para Loja: 9
Treinando modelo para Loja: 10
------------------------------
Arquivo 'previsoes_vendas.csv' gerado com 182500 linhas.
MAE Médio: 6.3507
